# 3.7 Merge ve Join

Bu notebook, PDS Handbook (TR) web sayfasının **Türkçe Jupyter karşılığıdır** — aynı açıklamalar, ders notları ve kod örnekleri.

| | |
|---|---|
| **Web sayfası** | `chapters/03-pandas/07-merge-and-join.html` |
| **Çalıştırma** | JupyterLab, VS Code veya Colab — hücreleri **yukarıdan aşağı** sırayla (`Shift+Enter`) |
| **Bağımlılık** | Kod hücreleri birbirine bağlıdır; hata alırsanız önce üsttekileri çalıştırın |

> **Kaynak:** Jake VanderPlas, *Python Data Science Handbook* — Türkçe ders uyarlaması



Orijinal: Combining Datasets: Merge and Join

Pandas'ın sunduğu önemli özelliklerden biri, veritabanlarında tanıdık olabileceğiniz yüksek performanslı bellek içi join ve merge işlemleridir. Ana arayüz pd.merge fonksiyonudur; pratikte nasıl çalıştığını örneklerle göreceğiz.

Kolaylık için önceki bölümdeki display fonksiyonunu standart içe aktarmalardan sonra yeniden tanımlayalım:


In [ ]:
# imports_display.py
import pandas as pd
import numpy as np

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        return '\n'.join(self.template.format(a, eval(a)._repr_html_())
                         for a in self.args)
    
    def __repr__(self):
        return '\n\n'.join(a + '\n' + repr(eval(a))
                           for a in self.args)



## İlişkisel Cebir

pd.merge'de uygulanan davranış, çoğu veritabanındaki işlemlerin kavramsal temeli olan ilişkisel cebirin bir alt kümesidir. Bu yaklaşımın gücü, herhangi bir veri kümesi üzerinde daha karmaşık bileşik işlemlerin yapı taşları olan temel işlemleri tanımlamasıdır.

Pandas, pd.merge ve Series/DataFrame join yönteminde bu yapı taşlarının birçoğunu uygular; farklı kaynaklardan veriyi verimli biçimde bağlamanızı sağlar.

## Join Türleri

pd.merge bire bir, çoka bir ve çoğa çok join türlerini destekler. Hepsi aynı arayüzle çağrılır; join türü girdi verisinin biçimine bağlıdır. Önce üç basit örneğe bakalım.

### Bire Bir Join

En basit merge türü bire bir join'dir; 3.6 Concat ve Append'deki sütun yönünde birleştirmeye benzer. Bir şirketteki çalışanlar hakkında iki DataFrame düşünelim:


In [ ]:
# df1_df2_employees.py
df1 = pd.DataFrame({'employee': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'group': ['Accounting', 'Engineering',
                              'Engineering', 'HR']})
df2 = pd.DataFrame({'employee': ['Lisa', 'Bob', 'Jake', 'Sue'],
                    'hire_date': [2004, 2008, 2012, 2014]})
display('df1', 'df2')



Bu bilgiyi tek bir DataFrame'de birleştirmek için pd.merge kullanılır:


In [ ]:
# pd_merge_df3.py
df3 = pd.merge(df1, df2)
df3



pd.merge, her iki çerçevede employee sütununu tanır ve otomatik olarak anahtar olarak kullanır. Sonuç iki girdinin bilgisini birleştirir. Sütunlardaki giriş sırası korunmayabilir; pd.merge bunu doğru hesaplar. Genelde merge indeksi atar (indeks üzerinden merge için left_index/right_index istisnası).

### Çoka Bir Join

Çoka bir join'de iki anahtar sütundan birinde yinelenen girişler vardır; sonuç DataFrame'i bu yinelenmeleri korur:


In [ ]:
# merge_many_to_one.py
df4 = pd.DataFrame({'group': ['Accounting', 'Engineering', 'HR'],
                    'supervisor': ['Carly', 'Guido', 'Steve']})
display('df3', 'df4', 'pd.merge(df3, df4)')



Sonuçta, girdi gerektirdiği yerlerde tekrarlanan “supervisor” bilgisi içeren ek bir sütun vardır.

### Çoğa Çok Join

Çoğa çok join kavramsal olarak kafa karıştırıcı olabilir ama iyi tanımlıdır. Her iki taraftaki anahtar sütunda yinelenme varsa sonuç çoğa çok merge olur. Bir grupla ilişkili becerileri gösteren çerçeveyle kişi başına becerileri kurtarabiliriz:


In [ ]:
# merge_many_to_many.py
df5 = pd.DataFrame({'group': ['Accounting', 'Accounting',
                              'Engineering', 'Engineering', 'HR', 'HR'],
                    'skills': ['math', 'spreadsheets', 'software', 'math',
                               'spreadsheets', 'organization']})
display('df1', 'df5', "pd.merge(df1, df5)")



Bu üç join türü diğer Pandas araçlarıyla geniş işlevsellik sağlar. Gerçek veri kümeleri nadiren bu kadar temizdir; pd.merge'in join'i nasıl ayarlayacağınıza dair seçeneklere geçelim.

## Merge Anahtarının Belirtilmesi

pd.merge'in varsayılan davranışını gördük: iki girdi arasında eşleşen sütun adlarını arar ve anahtar olarak kullanır. Sütun adları genelde bu kadar uyumlu olmaz; Pandas çeşitli seçenekler sunar.

### on Anahtar Sözcüğü

Anahtar sütun adını açıkça on ile verebilirsiniz (tek ad veya liste):


In [ ]:
# merge_on_employee.py
display('df1', 'df2', "pd.merge(df1, df2, on='employee')")



Bu seçenek yalnızca sol ve sağ DataFrame'de belirtilen sütun varsa çalışır.

### left_on ve right_on

Farklı sütun adlarıyla birleştirmek gerekebilir; örneğin çalışan adı “name” olarak etiketlenmiş olabilir. left_on ve right_on kullanılır:


In [ ]:
# merge_left_right_on.py
df3 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'salary': [70000, 80000, 120000, 90000]})
display('df1', 'df3', 'pd.merge(df1, df3, left_on="employee", right_on="name")')



Sonuçta gereksiz bir sütun kalabilir; DataFrame.drop() ile düşürülebilir:


In [ ]:
# merge_drop_name.py
pd.merge(df1, df3, left_on="employee", right_on="name").drop('name', axis=1)



### left_index ve right_index

Bazen sütun yerine indeks üzerinden birleştirmek istersiniz:


In [ ]:
# df1a_df2a_index.py
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')
display('df1a', 'df2a')



pd.merge() içinde left_index ve/veya right_index ile indeks anahtar olarak kullanılır:


In [ ]:
# merge_by_index.py
display('df1a', 'df2a',
        "pd.merge(df1a, df2a, left_index=True, right_index=True)")



Pandas, ek anahtar sözcük olmadan indeks tabanlı merge için DataFrame.join() yöntemini de sunar:


In [ ]:
# df1a_join_df2a.py
df1a.join(df2a)



İndeks ve sütunları karıştırmak için left_index ile right_on veya left_on ile right_index birleştirilebilir:


In [ ]:
# merge_left_index_right_on.py
display('df1a', 'df3', "pd.merge(df1a, df3, left_index=True, right_on='name')")



> **Not**
>

Tüm bu seçenekler çoklu indeks ve/veya çoklu sütunla da çalışır. Ayrıntılar için Pandas dokümantasyonundaki merge bölümüne bakın.

## Küme Aritmetiği ile Join

Önceki örneklerde göz ardı ettiğimiz konu: join'de kullanılan küme aritmetiği türü. Bir anahtar sütunda değer varken diğerinde yoksa ne olur?


In [ ]:
# df6_df7_food_drink.py
df6 = pd.DataFrame({'name': ['Peter', 'Paul', 'Mary'],
                    'food': ['fish', 'beans', 'bread']},
                   columns=['name', 'food'])
df7 = pd.DataFrame({'name': ['Mary', 'Joseph'],
                    'drink': ['wine', 'beer']},
                   columns=['name', 'drink'])
display('df6', 'df7', 'pd.merge(df6, df7)')



Yalnızca ortak “name” girişi Mary olan iki veri kümesi birleştirildi. Varsayılan sonuç iki girdinin kesişimidir — iç birleşim (inner join). how ile açıkça belirtilebilir (varsayılan "inner"):


In [ ]:
# merge_how_inner.py
pd.merge(df6, df7, how='inner')



how için diğer seçenekler 'outer', 'left' ve 'right''tır. Dış birleşim (outer join) girdi sütunlarının birleşimini döndürür; eksik değerler NA ile doldurulur:


In [ ]:
# merge_how_outer.py
display('df6', 'df7', "pd.merge(df6, df7, how='outer')")



Sol ve sağ join sırasıyla sol ve sağ girdinin girişleri üzerinden birleşim döndürür:


In [ ]:
# merge_how_left.py
display('df6', 'df7', "pd.merge(df6, df7, how='left')")



Çıktı satırları artık sol girdideki girişlere karşılık gelir. how='right' benzer şekilde sağ girdi için çalışır. Tüm seçenekler önceki join türlerine doğrudan uygulanabilir.

## Çakışan Sütun Adları: suffixes

İki girdide çakışan sütun adları olabilir:


In [ ]:
# df8_df9_rank.py
df8 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [1, 2, 3, 4]})
df9 = pd.DataFrame({'name': ['Bob', 'Jake', 'Lisa', 'Sue'],
                    'rank': [3, 1, 4, 2]})
display('df8', 'df9', 'pd.merge(df8, df9, on="name")')



Çıktıda iki rank sütunu çakışacağından merge otomatik olarak _x ve _y son eklerini ekler. Özel son ekler suffixes ile verilebilir:


In [ ]:
# merge_suffixes.py
pd.merge(df8, df9, on="name", suffixes=["_L", "_R"])



Bu son ekler tüm join desenlerinde ve birden fazla çakışan sütunda çalışır.

Bu desenler için 3.8 Agregasyon ve Gruplama'da ilişkisel cebire daha derin bakılır. Ayrıca Pandas dokümantasyonundaki Merge, join, concat and compare bölümüne bakın.

## Örnek: ABD Eyalet Verisi

Merge ve join en çok farklı kaynaklardan veri birleştirirken gündeme gelir. ABD eyaletleri ve nüfusları hakkında örnek veri kullanacağız. Veri dosyaları github.com/jakevdp/data-USstates adresindedir:


```
# Following are commands to download the data
# repo = "https://raw.githubusercontent.com/jakevdp/data-USstates/master"
# !cd data && curl -O {repo}/state-population.csv
# !cd data && curl -O {repo}/state-areas.csv
# !cd data && curl -O {repo}/state-abbrevs.csv
```


Üç veri kümesine Pandas read_csv ile bakalım (dosyalar data/ altında olmalıdır):


In [ ]:
# read_us_states_csv.py
pop = pd.read_csv('data/state-population.csv')
areas = pd.read_csv('data/state-areas.csv')
abbrevs = pd.read_csv('data/state-abbrevs.csv')

display('pop.head()', 'areas.head()', 'abbrevs.head()')



> **Not**
>

Bu bilgiyle 2010 nüfus yoğunluğuna göre ABD eyalet ve bölgelerini sıralamak isteyelim. Veri elimizde; birleştirmemiz gerekir.

Önce pop ile abbrevs arasında çoka bir merge yaparak tam eyalet adlarını alalım. pop'taki state/region ile abbrevs'teki abbreviation üzerinden birleştiririz; etiket uyumsuzluğunda veri kaybı olmasın diye how='outer':


In [ ]:
# merge_pop_abbrevs.py
merged = pd.merge(pop, abbrevs, how='outer',
                  left_on='state/region', right_on='abbreviation')
merged = merged.drop('abbreviation', axis=1) # drop duplicate info
merged.head()



Uyumsuzluk olup olmadığını null satırlara bakarak kontrol edelim:


In [ ]:
# merged_isnull.py
merged.isnull().any()



Bazı population değerleri null; hangileri olduğuna bakalım:


In [ ]:
# merged_null_population.py
merged[merged['population'].isnull()].head()



Null nüfus değerlerinin çoğu 2000 öncesi Porto Riko kayıtlarından geliyor olabilir. Daha önemlisi bazı yeni state girişleri de null — abbrevs anahtarında karşılık yok:


In [ ]:
# merged_null_state_region.py
merged.loc[merged['state'].isnull(), 'state/region'].unique()



Sorun: nüfus verisinde Porto Riko (PR) ve bir bütün olarak ABD (USA) var; kısaltma tablosunda yok. Hızlıca doldurabiliriz:


In [ ]:
# fill_pr_usa.py
merged.loc[merged['state/region'] == 'PR', 'state'] = 'Puerto Rico'
merged.loc[merged['state/region'] == 'USA', 'state'] = 'United States'
merged.isnull().any()



state sütununda artık null yok; devam edebiliriz.

Şimdi alan verisiyle state sütunu üzerinden birleştirelim:


In [ ]:
# merge_final_areas.py
final = pd.merge(merged, areas, on='state', how='left')
final.head()



Yine uyumsuzluk için null kontrolü:


In [ ]:
# final_isnull.py
final.isnull().any()



area sütununda null var; hangi bölgelerin atlandığına bakalım:


In [ ]:
# final_null_area_states.py
final['state'][final['area (sq. mi)'].isnull()].unique()



areas çerçevesinde ABD bütününün alanı yok. Toplam eyalet alanı eklenebilir; burada tüm ABD yoğunluğu tartışmamızla ilgili olmadığı için null satırları düşüyoruz:


In [ ]:
# final_dropna.py
final.dropna(inplace=True)
final.head()



Artık gereken veri hazır. 2010 yılı ve toplam nüfus dilimini query ile seçelim (3.12 Performans: eval ve query — NumExpr gerekebilir):


In [ ]:
# data2010_query.py
data2010 = final.query("year == 2010 & ages == 'total'")
data2010.head()



Nüfus yoğunluğunu hesaplayıp sıralayalım; eyalet üzerinden yeniden indeksleyerek:


In [ ]:
# density_compute.py
data2010.set_index('state', inplace=True)
density = data2010['population'] / data2010['area (sq. mi)']



In [ ]:
# density_sort.py
density.sort_values(ascending=False, inplace=True)
density.head()



Sonuç, 2010 nüfus yoğunluğuna göre (km² başına kişi) ABD eyaletleri, Washington DC ve Porto Riko sıralamasıdır. En yoğun bölge Washington DC; eyaletler arasında en yoğun New Jersey.

Listenin sonuna da bakalım:


In [ ]:
# density_tail.py
density.tail()



En seyrek eyalet açık ara Alaska — km² başına bir kişiden biraz fazla.

Bu tür veri birleştirme, gerçek dünya kaynaklarıyla soru yanıtlarken yaygındır. Bu örnek, öğrendiğimiz araçları birleştirerek veriden içgörü kazanmanın yollarından birini göstermiştir.

### 🧪 Şimdi deneyin

🧪 Şimdi deneyin
      İki küçük çerçeveyi pd.merge ile how='inner' ve how='outer' karşılaştırın:
          
      import pandas as pd
left = pd.DataFrame({'k': [1, 2, 3], 'v': ['a', 'b', 'c']})
right = pd.DataFrame({'k': [2, 3, 4], 'w': [10, 20, 30]})
print("inner:\n", pd.merge(left, right, on='k', how='inner'))
print("\nouter:\n", pd.merge(left, right, on='k', how='outer'))

> **Not**
>
